In [1]:
# # Core imports man
# import sys
# import os
# import random
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import scipy
# import time

# # Numerical libraries
# from scipy.integrate import solve_ivp, odeint, cumulative_trapezoid
# from scipy.interpolate import (
#     UnivariateSpline, splrep, splev, CubicSpline,
#     interp1d, PchipInterpolator, InterpolatedUnivariateSpline
# )

# import numdifftools as nd

# # Random number generator (GSL)
# import pygsl.rng

# # custom InflationModels code to path the one below is for wkb approximation method
# sys.path.append(
#     '/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/InflationModels'
# )


# # Local modules from InflationModels
# from MacroDefinitions import *
# from calcpath import *
# from int_de import *
# if SPECTRUM:
#     from spectrum_OG_nanoscale_nodiagnostics import *

# # ========================
# # GLOBAL SETTINGS
# # ========================
# NEQS = 9
# # SPECTRUM = False
# SPECTRUM = True
# SAVEPATHS = True

# NMAX = 1.2
# NMIN = 0.3

# LAM6_BASE = 6.12536e-10
# LAM6_DELTA = 1.5e-10

# LAM6_MIN = LAM6_BASE - LAM6_DELTA  
# LAM6_MAX = LAM6_BASE + LAM6_DELTA 

# #4.6e-10

# NUM_LAM6_GRID = 1

# #Setup for more models
# # lam6_set = np.linspace(LAM6_MIN, LAM6_MAX, num=NUM_LAM6_GRID)
# # lam6_set = np.sort(np.append(lam6_set, [0.0]))

# # Base set just 1
# # lam6_set = np.linspace(6.12536e-10, 1.5e-10, num=NUM_LAM6_GRID) 
# lam6_set = [6.12536e-10, 0, 6.12536e-09, 6.12536e-08, 6.12536e-07, 6.12536e-06, 6.12536e-11, 6.12536e-12, 6.12536e-13, 6.12536e-14, 6.12536e-15, 6.12536e-16, 6.12536e-17, 6.12536e-18, 6.12536e-19, 6.12536e-20, 6.12536e-23, 6.12536e-26, 6.12536e-29, 6.12536e-32] 
# derivs1 = derivs

# print(f"Total models: {len(lam6_set)}")
# # print(f"Base λ6={LAM6_BASE:.6e} included: {LAM6_BASE in lam6_set}")

# # This sets how many nontrivial models you are running
# NUMPOINTS = 1

# # This will give us the min and max number of e-folds we are looking for
# NUMEFOLDSMAX = 65.0
# NUMEFOLDSMIN = 57.0

# # Here we are writing out our output files
# BASE_OUTDIR = "/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs9"

# # RNG initialization process
# my_random = pygsl.rng.ranlxd2()
# my_random.set(0)
# np.random.seed(0)

# # ========================
# # Support Functions
# # ========================
# class Calc:
#     def __init__(self):
#         self.Y = np.zeros(NEQS, dtype=float, order='C')
#         self.initY = np.zeros(NEQS, dtype=float, order='C')
#         self.ret = ""
#         self.npoints = 0
#         self.Nefolds = 0.0


# def pick_init_vals(lam6):
#     init_vals = np.zeros(NEQS, dtype=float, order='C')
#     init_vals[0] = 5.5
#     init_vals[1] = 1.0
#     init_vals[2] = 0.000209237
#     init_vals[3] = -0.0342419
#     init_vals[4] = 0.000278972
#     init_vals[5] = -4.60971e-06
#     init_vals[6] = 6.87065e-08
#     init_vals[7] = -8.92461e-9
#     init_vals[8] = lam6

# #     init_Nefolds = my_random.uniform() * (NUMEFOLDSMAX - NUMEFOLDSMIN) + NUMEFOLDSMIN
#     init_Nefolds = 60
#     return init_vals, init_Nefolds


# def we_should_calc_spec(y):
#     return (specindex(y) > NMIN and specindex(y) < NMAX)


# def we_should_save_path(retval, save, pointcount, printevery):
#     return (retval == "nontrivial") and (not save) and (pointcount % printevery == 0)


# def save_path(y, N, kount, fname):
#     with open(fname, "w") as outfile:
#         for i in range(kount):
#             for j in range(NEQS):
#                 outfile.write("%le " % y[j, i])
#             outfile.write("%lf " % N[i])

#             V = (3. / (8. * np.pi)) * y[1, i] * y[1, i] * (1. - y[2, i] / 3.)
#             outfile.write("%le %le\n" % (V, (V * y[2, i]) / (3. - y[2, i])))


# # ========================
# # Main Loop
# # ========================
# def run_neqs9_models():
#     summary_records = []

#     for lam6 in lam6_set:
#         print(f"\n=== Running λ6 = {lam6:.1e} ===")

#         OUTDIR = f"{BASE_OUTDIR}/lam6_{lam6:.1e}"
#         os.makedirs(OUTDIR, exist_ok=True)
#         OUTFILE1_NAME = f"{OUTDIR}/test_nr_neqs{NEQS}.dat"
#         OUTFILE2_NAME = f"{OUTDIR}/test_esigma_neqs{NEQS}.dat"

#         try:
#             outfile1 = open(OUTFILE1_NAME, "w")
#             outfile2 = open(OUTFILE2_NAME, "w")
#         except IOError as e:
#             print("Could not open output files: ", e)
#             sys.exit()

#         if SPECTRUM:
#             u_s = np.empty((2, knos))
#             u_t = np.empty((2, knos))
#             y_final = np.empty(NEQS + 1)
#             spec_count = 0

#         calc = Calc()
#         iters = 0
#         points = 0
#         outcount = 0
#         asymcount = 0
#         nontrivcount = 0
#         insuffcount = 0
#         noconvcount = 0
#         badncount = 0
#         errcount = 0
#         savedone = 0

#         while nontrivcount < NUMPOINTS:
#             iters += 1
#             if iters > 200:
#                 break

#             if iters % 100 == 0:
#                 print(f"  Iter {iters}, nontriv={nontrivcount}")

#             yinit, calc.Nefolds = pick_init_vals(lam6)
#             y = yinit.copy()

#             path = np.array([[]])
#             N = np.array([])

#             t0 = time.perf_counter()
#             calc.ret = calcpath(calc.Nefolds, y, path, N, calc)
#             t1 = time.perf_counter()
#             print(f"calcpath runtime: {t1 - t0:.4f} s")
#             print(f"  -> {calc.ret}")

#             print("\n=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===")
#             print(f"Initial values (yinit): {yinit}")
#             print(f"  φ0 = {yinit[0]:.6e}")
#             print(f"  H0 = {yinit[1]:.6e}")
#             print(f"  ε0 = {yinit[2]:.6e}")
#             print(f"  σ0 = {yinit[3]:.6e}")
#             print(f"  λ₂ = {yinit[4]:.6e}")
#             if len(yinit) > 5:
#                 print(f"  λ₃ = {yinit[5]:.6e}")
#             if len(yinit) > 6:
#                 print(f"  λ₄ = {yinit[6]:.6e}")
#             if len(yinit) > 7:
#                 print(f"  λ₅ = {yinit[7]:.6e}")
#             print(f"Initial Nefolds = {calc.Nefolds:.3f}")

#             try:
#                 import pygsl.odeiv as odeiv
#                 s = odeiv.step_rk4(len(yinit), derivs1)
#                 c = odeiv.control_y_new(s, 1e-8, 1e-8)
#                 print(f"GSL integrator class: {s.__class__.__name__}")
#                 print(f"Expected tolerances: atol=1e-8, rtol=1e-8")
#             except Exception as e:
#                 print("Could not check GSL integrator:", e)

#             print("===============================================\n")

#             if calc.ret == "asymptote":
#                 asymcount += 1
#                 if asymcount > 100:
#                     print("Too many asymptotes, stopping")
#                     break
#                 continue

#             if calc.ret == "nontrivial":
#                 r = tsratio(y)
#                 ns = specindex(y)
#                 alpha_s = dspecindex(y)
#                 outfile1.write(f"{r:.10f} {ns:.10f} {alpha_s:.10f}\n")
#                 outfile1.flush()

#                 for i in range(NEQS):
#                     outfile2.write("%le " % y[i])
#                 outfile2.write("%f\n" % calc.Nefolds)
#                 outfile2.flush()

#                 points += 1
#                 savedone = 0
#                 nontrivcount += 1

#                 if SPECTRUM and we_should_calc_spec(y):
#                     print(f"  ns = {specindex(y):.3f}")
#                     print(f"  -> Evaluating spectrum {spec_count}")

#                     y_final[:NEQS] = path[:NEQS, 3]
#                     y_final[NEQS] = N[3]

#                     t0 = time.perf_counter()
#                     spectrum_status = spectrum(
#                         y_final, y, u_s, u_t, calc.Nefolds,
#                         derivs1, scalarsys, tensorsys
#                     )
#                     t1 = time.perf_counter()
#                     print(f"spectrum runtime: {t1 - t0:.4f} s")

#                     if spectrum_status:
#                         errcount += 1

#                     spec_s_name = f"{OUTDIR}/spec_s{spec_count:03d}_neqs{NEQS}.dat"
#                     spec_t_name = f"{OUTDIR}/spec_t{spec_count:03d}_neqs{NEQS}.dat"
#                     np.savetxt(spec_s_name, u_s[:, :knos].T)
#                     np.savetxt(spec_t_name, u_t[:, :knos].T)
#                     spec_count += 1

#                 if SPECTRUM:
#                     print(f"  -> Before normalization: y[1] = {y[1]:.6e}")
#                     for j in range(calc.npoints):
#                         path[0, j] = path[0, j] - path[0, calc.npoints - 1]
#                         path[1, j] = path[1, j] * y[1]

#                     print(f"  -> After normalization: max(path[1,:]) = {np.max(path[1,:]):.6e}")

#                 path_name = f"{OUTDIR}/path_neqs{NEQS}_lam6{lam6:.1e}_{outcount:03d}.dat"
#                 print(f"  -> Saving path {path_name}")
#                 save_path(path, N, calc.npoints, path_name)
#                 outcount += 1

#                 summary_records.append({
#                     "lam6": lam6,
#                     "r": r,
#                     "n_s": ns,
#                     "alpha_s": alpha_s,
#                     "Nefolds": calc.Nefolds
#                 })

#             elif calc.ret == "insuff":
#                 insuffcount += 1
#             elif calc.ret == "noconverge":
#                 noconvcount += 1
#             else:
#                 errcount += 1

#         outfile1.close()
#         outfile2.close()

#     summary_df = pd.DataFrame(summary_records)
#     summary_file = f"{BASE_OUTDIR}/neqs{NEQS}_summary.csv"
#     summary_df.to_csv(summary_file, index=False)
#     print(f"\nSummary written to {summary_file}")


# print('env', sys.executable)

# import platform, numpy, scipy, pandas, matplotlib
# print(f"Python: {platform.python_version()}")
# print(f"CPU Archachitecture: {platform.machine()}")
# print(f"NumPy: {numpy.__version__}")
# print(f"SciPy: {scipy.__version__}")
# print(f"pandas: {pandas.__version__}")
# print(f"matplotlib: {matplotlib.__version__}")

# %time run_neqs9_models()

Total models: 20
env /Users/epmeador/opt/anaconda3/bin/python
Python: 3.8.5
CPU Archachitecture: x86_64
NumPy: 1.24.4
SciPy: 1.10.1
pandas: 1.1.3
matplotlib: 3.3.2

=== Running λ6 = 6.1e-10 ===
calcpath runtime: 0.0352 s
  -> nontrivial

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-10]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

  ns = 0.971
  -> Evaluating spectrum 0
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91

189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
spectrum runtime: 55.0050 s
  -> Before normalization: y[1] = 1.012817e-06
  -> After normalization: max(path[1,:]) = 1.017876e-06
  -> Saving path /Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs9/lam6_6.1e-07/path_neqs9_lam66.1e-07_000.dat

=== Running λ6 = 6.1e-06 ===
calcpath runtime: 0.0741 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

calcpath runtime: 0.0803 s
  -> insuff

=== IN

calcpath runtime: 0.0789 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

calcpath runtime: 0.0802 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

calcpa

calcpath runtime: 0.0785 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

calcpath runtime: 0.0765 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

calcpa

calcpath runtime: 0.0762 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

calcpath runtime: 0.0818 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

calcpa

...

calcpath runtime: 0.0774 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

calcpath runtime: 0.0821 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

calcpa

calcpath runtime: 0.0764 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

calcpath runtime: 0.0756 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

calcpa

calcpath runtime: 0.0775 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

calcpath runtime: 0.0774 s
  -> insuff

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -8.92461e-09  6.12536e-06]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₃ = -4.609710e-06
  λ₄ = 6.870650e-08
  λ₅ = -8.924610e-09
Initial Nefolds = 60.000
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

  Iter

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
spectrum runtime: 55.2491 s
  -> Before normalization: y[1] = 1.165412e-06
  -> After normalization: max(path[1,:]) = 1.166811e-06
  -> Saving path /Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/h

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
spectrum runtime: 54.1781 s
  -> Before normalization: y[1] = 1.165413e-06
  -> After normalization: max(path[1,:]) = 1.166812e-06
  -> Saving path /Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/h

If I want to modify this so that I am able to run it and accoujnt for original number of e-fold I would do so like this:

In [1]:
# Core imports man
import sys
import os
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import time

from scipy.integrate import solve_ivp, odeint, cumulative_trapezoid
from scipy.interpolate import (
    UnivariateSpline, splrep, splev, CubicSpline,
    interp1d, PchipInterpolator, InterpolatedUnivariateSpline
)

import numdifftools as nd
import pygsl.rng

sys.path.append(
    "/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/InflationModels"
)

# ========================
# GLOBAL SETTINGS
# ========================
NEQS = 9
SPECTRUM = True
SAVEPATHS = True

NMAX = 0.971
NMIN = 0.96

LAM6_BASE = 6.1e-10

# Keep your current tight λ6 range
LAM6_DELTA = 1.5e-10
LAM6_MIN = LAM6_BASE - LAM6_DELTA
LAM6_MAX = LAM6_BASE + LAM6_DELTA


NUM_LAM6_GRID = 100

# Kinney Range
# LAM6_MIN = -5e-10 #-5e-6
# LAM6_MAX = 5e-10 #5e-6

lam6_set = np.random.uniform(LAM6_MIN, LAM6_MAX, size=NUM_LAM6_GRID)
lam6_set = np.sort(np.append(lam6_set, [LAM6_BASE])) #used to also do a zero

print(f"Total models: {len(lam6_set)}")
print(f"Base λ6={LAM6_BASE:.6e} included: {LAM6_BASE in lam6_set}")

NUMPOINTS = 1

NUMEFOLDSMAX = 65.0
NUMEFOLDSMIN = 57.0

BASE_PATH_ROOT = "/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests"
BASE_OUTDIR = f"{BASE_PATH_ROOT}/neqs{NEQS}"

my_random = pygsl.rng.ranlxd2()
my_random.set(0)
np.random.seed(0)

# Local modules
from MacroDefinitions import *
from calcpath import *
from int_de import *

if SPECTRUM:
    from spectrum_OG_nanoscale_nodiagnostics import *


class Calc:
    def __init__(self):
        self.Y = np.zeros(NEQS, dtype=float, order="C")
        self.initY = np.zeros(NEQS, dtype=float, order="C")
        self.ret = ""
        self.npoints = 0
        self.Nefolds = 0.0


def pick_init_vals(lam6):
    init_vals = np.zeros(NEQS, dtype=float, order="C")

    init_vals[0] = 5.5 / np.sqrt(8 * np.pi)  # phi0
    init_vals[1] = 1.0                       # H0
    init_vals[2] = 0.000209237               # epsilon0
    init_vals[3] = -0.0342419                # sigma0
    init_vals[4] = 0.000278972               # lambda2
    init_vals[5] = -4.60971e-6               # lambda3
    init_vals[6] = 6.87065e-08               # lambda4
    init_vals[7] = -8.92461e-9               # lambda5
    init_vals[8] = lam6                      # lambda6

    init_Nefolds = 60
    return init_vals, init_Nefolds


def we_should_calc_spec(y):
    return (specindex(y) > NMIN and specindex(y) < NMAX)


def we_should_save_path(retval, save, pointcount, printevery):
    return (retval == "nontrivial") and (not save) and (pointcount % printevery == 0)


def save_path(y, N, kount, fname):
    with open(fname, "w") as outfile:
        for i in range(kount):
            for j in range(NEQS):
                outfile.write("%le " % y[j, i])

            outfile.write("%lf " % N[i])

            V = (3.0 / (8.0 * np.pi)) * y[1, i] * y[1, i] * (1.0 - y[2, i] / 3.0)

            outfile.write(
                "%le %le\n" %
                (
                    V,
                    (V * y[2, i]) / (3.0 - y[2, i]),
                )
            )


def run_neqs9_models(clean_output=True):

    TARGET_ACCEPTED = 5
    MAX_TRIALS = 100000

    summary_records = []

    if clean_output and os.path.exists(BASE_OUTDIR):
        print(f"Removing old output directory:\n{BASE_OUTDIR}")
        shutil.rmtree(BASE_OUTDIR)

    os.makedirs(BASE_OUTDIR, exist_ok=True)

    accepted_count = 0
    trial_count = 0

    rejected_asymptote = 0
    rejected_bad_ns = 0
    rejected_other = 0
    spectrum_error_count = 0
    duplicate_dir_count = 0

    while accepted_count < TARGET_ACCEPTED and trial_count < MAX_TRIALS:

        trial_count += 1

#         if trial_count == 1:
#             lam6 = LAM6_BASE
#         elif trial_count == 2:
#             lam6 = 0.0
#         else:
#             lam6 = np.random.uniform(LAM6_MIN, LAM6_MAX)
     
    
        if trial_count == 1:
            lam6 = LAM6_BASE
#         elif trial_count == 2:
#             lam6 = 0.0
        else:
            lam6 = np.random.uniform(LAM6_MIN, LAM6_MAX)


        print("\n" + "=" * 70)
        print(f"Trial {trial_count} | accepted {accepted_count}/{TARGET_ACCEPTED}")
        print(f"Trying λ6 = {lam6:.10e}")

        calc = Calc()

        yinit, calc.Nefolds = pick_init_vals(lam6)
        y = yinit.copy()

        path = np.array([[]])
        N = np.array([])

        t0 = time.perf_counter()
        calc.ret = calcpath(calc.Nefolds, y, path, N, calc)
        t1 = time.perf_counter()

        if calc.npoints > 5:
            print("\nDEBUG N values:")
            print("  N[0]   =", N[0])
            print("  N[3]   =", N[3])
            print("  N[end] =", N[calc.npoints - 1])
            print("  Nefolds target =", calc.Nefolds)
        else:
            print("WARNING: not enough N points to debug")

        print(f"calcpath runtime: {t1 - t0:.4f} s")
        print(f"calc.ret = {calc.ret}")

        if calc.ret == "asymptote":
            rejected_asymptote += 1
            print("REJECTED: asymptote")
            continue

        if calc.ret != "nontrivial":
            rejected_other += 1
            print(f"REJECTED: {calc.ret}")
            continue

        r = tsratio(y)
        ns = specindex(y)
        alpha_s = dspecindex(y)

        print("Candidate observables:")
        print(f"  r       = {r:.10e}")
        print(f"  ns      = {ns:.10f}")
        print(f"  alpha_s = {alpha_s:.10e}")

        if not (NMIN < ns < NMAX):
            rejected_bad_ns += 1
            print(f"REJECTED: ns={ns:.10f} outside ({NMIN}, {NMAX})")
            continue

        accepted_count += 1

        print("\n*** ACCEPTED MODEL ***")
        print(f"accepted #{accepted_count}")
        print(f"λ6 = {lam6:.10e}")
        print(f"ns = {ns:.10f}")

        OUTDIR = f"{BASE_OUTDIR}/lam6_{lam6:.10e}"

        if os.path.exists(OUTDIR):
            duplicate_dir_count += 1
            OUTDIR = f"{BASE_OUTDIR}/lam6_{lam6:.10e}_trial_{trial_count:06d}"

        os.makedirs(OUTDIR, exist_ok=False)

        OUTFILE1_NAME = f"{OUTDIR}/test_nr_neqs{NEQS}.dat"
        OUTFILE2_NAME = f"{OUTDIR}/test_esigma_neqs{NEQS}.dat"

        with open(OUTFILE1_NAME, "w") as outfile1:
            outfile1.write(f"{r:.10f} {ns:.10f} {alpha_s:.10f}\n")

        with open(OUTFILE2_NAME, "w") as outfile2:
            for i in range(NEQS):
                outfile2.write("%le " % y[i])
            outfile2.write("%f\n" % calc.Nefolds)

        if SPECTRUM:
            u_s = np.empty((2, knos))
            u_t = np.empty((2, knos))
            y_final = np.empty(NEQS + 1)

            if calc.npoints <= 3:
                print("WARNING: not enough path points for spectrum. Skipping spectrum.")
            else:
                y_final[:NEQS] = path[:NEQS, 3]
                y_final[NEQS] = N[3]

                print("Evaluating spectrum for accepted model...")

                t0 = time.perf_counter()
                spectrum_status = spectrum(
                    y_final,
                    y,
                    u_s,
                    u_t,
                    calc.Nefolds,
                    derivs1,
                    scalarsys,
                    tensorsys,
                )
                t1 = time.perf_counter()

                print(f"spectrum runtime: {t1 - t0:.4f} s")

                if spectrum_status:
                    spectrum_error_count += 1
                    print("WARNING: spectrum returned an error/status flag.")

                np.savetxt(f"{OUTDIR}/spec_s_neqs{NEQS}.dat", u_s[:, :knos].T)
                np.savetxt(f"{OUTDIR}/spec_t_neqs{NEQS}.dat", u_t[:, :knos].T)

        if SPECTRUM:
            print(f"Before path normalization: y[1] = {y[1]:.6e}")

            for j in range(calc.npoints):
                path[0, j] = path[0, j] - path[0, calc.npoints - 1]
                path[1, j] = path[1, j] * y[1]

            print(f"After path normalization: max(path[1,:]) = {np.max(path[1, :]):.6e}")

        path_name = f"{OUTDIR}/path_neqs{NEQS}_lam6_{lam6:.10e}.dat"
        save_path(path, N, calc.npoints, path_name)

        print("\nDEBUG original calcpath end:")
        print("  original_end_index      =", getattr(calc, "original_end_index", None))
        print("  original_N_end          =", getattr(calc, "original_N_end", None))
        print("  original_N_before_end   =", getattr(calc, "original_N_before_end", None))
        print("  original_N_after_end    =", getattr(calc, "original_N_after_end", None))
        print("  original_eps_end        =", getattr(calc, "original_eps_end", None))
        print("  spectrum_N_start N[3]   =", N[3])
        print("  path_N_end N[-1]        =", N[calc.npoints - 1])

        summary_records.append({
            "accepted_index": accepted_count,
            "trial_index": trial_count,
            "lam6": lam6,
            "r": r,
            "n_s": ns,
            "alpha_s": alpha_s,
            "Nefolds": calc.Nefolds,

            "original_end_index": getattr(calc, "original_end_index", np.nan),
            "original_N_end": getattr(calc, "original_N_end", np.nan),
            "original_eps_end": getattr(calc, "original_eps_end", np.nan),

            "spectrum_N_start": N[3],
            "path_N_end": N[calc.npoints - 1],

            "calc_ret": calc.ret,
            "outdir": OUTDIR,
        })

    summary_df = pd.DataFrame(summary_records)
    summary_file = f"{BASE_OUTDIR}/neqs{NEQS}_summary.csv"
    summary_df.to_csv(summary_file, index=False)

    print("\n" + "=" * 70)
    print("DONE")
    print(f"Accepted viable nontrivial models: {accepted_count}")
    print(f"Total trials: {trial_count}")
    print(f"Rejected asymptotes: {rejected_asymptote}")
    print(f"Rejected bad ns: {rejected_bad_ns}")
    print(f"Rejected other: {rejected_other}")
    print(f"Spectrum error count: {spectrum_error_count}")
    print(f"Duplicate directory count: {duplicate_dir_count}")
    print(f"Summary written to:\n{summary_file}")

    if accepted_count < TARGET_ACCEPTED:
        print(
            f"WARNING: only found {accepted_count}/{TARGET_ACCEPTED} accepted models "
            f"before hitting MAX_TRIALS={MAX_TRIALS}."
        )


%time run_neqs9_models()

Total models: 101
Base λ6=6.100000e-10 included: True
Removing old output directory:
/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs9

Trial 1 | accepted 0/5
Trying λ6 = 6.1000000000e-10

DEBUG N values:
  N[0]   = 1.146444335769047e-06
  N[3]   = 0.00015614644433576904
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0321 s
calc.ret = nontrivial
Candidate observables:
  r       = 2.5291091526e-03
  ns      = 0.9706773294
  alpha_s = -4.3334678021e-04

*** ACCEPTED MODEL ***
accepted #1
λ6 = 6.1000000000e-10
ns = 0.9706773294
Evaluating spectrum for accepted model...
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
1

163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
spectrum runtime: 52.3997 s
Before path normalization: y[1] = 1.119658e-06
After path normalization: max(path[1,:]) = 1.121503e-06

DEBUG original calcpath end:
  original_end_index      = 143
  original_N_end          = 949.0806775165963
  original_N_before_end   = 949.0806775318841
  original_N_after_end    = 949.0806775013085
  original_eps_end        = 1.0000000100416695
  spectrum_N_start N[3]   = 0.00015604307696599426
  path_N_end N[-1]        = 60.0

DONE
Accepted viable nontrivial models: 5
Total trials: 5
Rejected asymptotes: 0
Rejected bad ns: 0
Rejected other: 0
Spectrum error count: 0
Duplicate directory count: 0
Summary written to:
/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs9/neqs9_s

Now, if I want to run this and see what it looks like when lambda5 and lambda6 are random, I would do:

In [1]:
# Core imports man
import sys
import os
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import time

from scipy.integrate import solve_ivp, odeint, cumulative_trapezoid
from scipy.interpolate import (
    UnivariateSpline, splrep, splev, CubicSpline,
    interp1d, PchipInterpolator, InterpolatedUnivariateSpline
)

import numdifftools as nd
import pygsl.rng

sys.path.append(
    "/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/InflationModels"
)

# ========================
# GLOBAL SETTINGS
# ========================

NEQS = 9
SPECTRUM = True
SAVEPATHS = True

NMAX = 0.971
NMIN = 0.96

LAM5_BASE = -8.92461e-9
LAM6_BASE = 6.1e-10

# Random ranges
LAM5_MIN = -5e-5
LAM5_MAX = 5e-5

#trying super small range for now not kinney one
LAM6_MIN = -5e-9
LAM6_MAX = 5e-9

NUM_LAM_GRID = 100

lam5_set = np.random.uniform(LAM5_MIN, LAM5_MAX, size=NUM_LAM_GRID)
lam5_set = np.sort(np.append(lam5_set, [LAM5_BASE]))

lam6_set = np.random.uniform(LAM6_MIN, LAM6_MAX, size=NUM_LAM_GRID)
lam6_set = np.sort(np.append(lam6_set, [LAM6_BASE]))

print(f"Total λ5 trial pool: {len(lam5_set)}")
print(f"Total λ6 trial pool: {len(lam6_set)}")
print(f"Base λ5={LAM5_BASE:.6e} included: {LAM5_BASE in lam5_set}")
print(f"Base λ6={LAM6_BASE:.6e} included: {LAM6_BASE in lam6_set}")

NUMPOINTS = 1

NUMEFOLDSMAX = 65.0
NUMEFOLDSMIN = 57.0

BASE_PATH_ROOT = (
    "/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/"
    "inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests"
)

# BASE_OUTDIR = f"{BASE_PATH_ROOT}/neqs{NEQS}_random_lam5_lam6"
BASE_OUTDIR = f"{BASE_PATH_ROOT}/neqs{NEQS}"

my_random = pygsl.rng.ranlxd2()
my_random.set(0)
np.random.seed(0)

# Local modules
from MacroDefinitions import *
from calcpath import *
from int_de import *

if SPECTRUM:
    from spectrum_OG_nanoscale_nodiagnostics import *


class Calc:
    def __init__(self):
        self.Y = np.zeros(NEQS, dtype=float, order="C")
        self.initY = np.zeros(NEQS, dtype=float, order="C")
        self.ret = ""
        self.npoints = 0
        self.Nefolds = 0.0


def pick_init_vals(lam5, lam6):
    init_vals = np.zeros(NEQS, dtype=float, order="C")

    init_vals[0] = 5.5 / np.sqrt(8 * np.pi)  # phi0
    init_vals[1] = 1.0                       # H0
    init_vals[2] = 0.000209237               # epsilon0
    init_vals[3] = -0.0342419                # sigma0
    init_vals[4] = 0.000278972               # lambda2
    init_vals[5] = -4.60971e-6               # lambda3
    init_vals[6] = 6.87065e-08                 # lambda4 fixed
    init_vals[7] = lam5                      # lambda5 randomized
    init_vals[8] = lam6                      # lambda6 randomized

    init_Nefolds = 60
    return init_vals, init_Nefolds


def we_should_calc_spec(y):
    return (specindex(y) > NMIN and specindex(y) < NMAX)


def we_should_save_path(retval, save, pointcount, printevery):
    return (retval == "nontrivial") and (not save) and (pointcount % printevery == 0)


def save_path(y, N, kount, fname):
    with open(fname, "w") as outfile:
        for i in range(kount):
            for j in range(NEQS):
                outfile.write("%le " % y[j, i])

            outfile.write("%lf " % N[i])

            V = (
                (3.0 / (8.0 * np.pi))
                * y[1, i]
                * y[1, i]
                * (1.0 - y[2, i] / 3.0)
            )

            outfile.write(
                "%le %le\n"
                % (
                    V,
                    (V * y[2, i]) / (3.0 - y[2, i]),
                )
            )


def run_neqs9_lam5_lam6_models(clean_output=True):

    TARGET_ACCEPTED = 5
    MAX_TRIALS = 100000

    summary_records = []

    if clean_output and os.path.exists(BASE_OUTDIR):
        print(f"Removing old output directory:\n{BASE_OUTDIR}")
        shutil.rmtree(BASE_OUTDIR)

    os.makedirs(BASE_OUTDIR, exist_ok=True)

    accepted_count = 0
    trial_count = 0

    rejected_asymptote = 0
    rejected_bad_ns = 0
    rejected_other = 0
    spectrum_error_count = 0
    duplicate_dir_count = 0

    while accepted_count < TARGET_ACCEPTED and trial_count < MAX_TRIALS:

        trial_count += 1

        if trial_count == 1:
            lam5 = LAM5_BASE
            lam6 = LAM6_BASE
        else:
            lam5 = np.random.uniform(LAM5_MIN, LAM5_MAX)
            lam6 = np.random.uniform(LAM6_MIN, LAM6_MAX)

        print("\n" + "=" * 70)
        print(f"Trial {trial_count} | accepted {accepted_count}/{TARGET_ACCEPTED}")
        print(f"Trying λ5 = {lam5:.10e}")
        print(f"Trying λ6 = {lam6:.10e}")

        calc = Calc()

        yinit, calc.Nefolds = pick_init_vals(lam5, lam6)
        y = yinit.copy()

        path = np.array([[]])
        N = np.array([])

        t0 = time.perf_counter()
        calc.ret = calcpath(calc.Nefolds, y, path, N, calc)
        t1 = time.perf_counter()

        if calc.npoints > 5:
            print("\nDEBUG N values:")
            print("  N[0]   =", N[0])
            print("  N[3]   =", N[3])
            print("  N[end] =", N[calc.npoints - 1])
            print("  Nefolds target =", calc.Nefolds)
        else:
            print("WARNING: not enough N points to debug")

        print(f"calcpath runtime: {t1 - t0:.4f} s")
        print(f"calc.ret = {calc.ret}")

        if calc.ret == "asymptote":
            rejected_asymptote += 1
            print("REJECTED: asymptote")
            continue

        if calc.ret != "nontrivial":
            rejected_other += 1
            print(f"REJECTED: {calc.ret}")
            continue

        r = tsratio(y)
        ns = specindex(y)
        alpha_s = dspecindex(y)

        print("Candidate observables:")
        print(f"  r       = {r:.10e}")
        print(f"  ns      = {ns:.10f}")
        print(f"  alpha_s = {alpha_s:.10e}")

        if not (NMIN < ns < NMAX):
            rejected_bad_ns += 1
            print(f"REJECTED: ns={ns:.10f} outside ({NMIN}, {NMAX})")
            continue

        accepted_count += 1

        print("\n*** ACCEPTED MODEL ***")
        print(f"accepted #{accepted_count}")
        print(f"λ5 = {lam5:.10e}")
        print(f"λ6 = {lam6:.10e}")
        print(f"ns = {ns:.10f}")

#         OUTDIR = f"{BASE_OUTDIR}/lam5_{lam5:.10e}_lam6_{lam6:.10e}" #probaly more proper
        OUTDIR = f"{BASE_OUTDIR}/lam6_{lam6:.10e}"


        if os.path.exists(OUTDIR):
            duplicate_dir_count += 1
#             OUTDIR = (
#                 f"{BASE_OUTDIR}/lam5_{lam5:.10e}_"
#                 f"lam6_{lam6:.10e}_trial_{trial_count:06d}"
#             )
        
            OUTDIR = f"{BASE_OUTDIR}/lam6_{lam6:.10e}_trial_{trial_count:06d}"


        os.makedirs(OUTDIR, exist_ok=False)

        OUTFILE1_NAME = f"{OUTDIR}/test_nr_neqs{NEQS}.dat"
        OUTFILE2_NAME = f"{OUTDIR}/test_esigma_neqs{NEQS}.dat"
        

        with open(OUTFILE1_NAME, "w") as outfile1:
            outfile1.write(f"{r:.10f} {ns:.10f} {alpha_s:.10f}\n")

        with open(OUTFILE2_NAME, "w") as outfile2:
            for i in range(NEQS):
                outfile2.write("%le " % y[i])
            outfile2.write("%f\n" % calc.Nefolds)

        if SPECTRUM:
            u_s = np.empty((2, knos))
            u_t = np.empty((2, knos))
            y_final = np.empty(NEQS + 1)

            if calc.npoints <= 3:
                print("WARNING: not enough path points for spectrum. Skipping spectrum.")
            else:
                y_final[:NEQS] = path[:NEQS, 3]
                y_final[NEQS] = N[3]

                print("Evaluating spectrum for accepted model...")

                t0 = time.perf_counter()
                spectrum_status = spectrum(
                    y_final,
                    y,
                    u_s,
                    u_t,
                    calc.Nefolds,
                    derivs1,
                    scalarsys,
                    tensorsys,
                )
                t1 = time.perf_counter()

                print(f"spectrum runtime: {t1 - t0:.4f} s")

                if spectrum_status:
                    spectrum_error_count += 1
                    print("WARNING: spectrum returned an error/status flag.")

                np.savetxt(
                    f"{OUTDIR}/spec_s_neqs{NEQS}.dat",
                    u_s[:, :knos].T,
                )

                np.savetxt(
                    f"{OUTDIR}/spec_t_neqs{NEQS}.dat",
                    u_t[:, :knos].T,
                )

        if SPECTRUM:
            print(f"Before path normalization: y[1] = {y[1]:.6e}")

            for j in range(calc.npoints):
                path[0, j] = path[0, j] - path[0, calc.npoints - 1]
                path[1, j] = path[1, j] * y[1]

            print(
                f"After path normalization: "
                f"max(path[1,:]) = {np.max(path[1, :]):.6e}"
            )

#         path_name = (
#             f"{OUTDIR}/path_neqs{NEQS}_"
#             f"lam5_{lam5:.10e}_"
#             f"lam6_{lam6:.10e}.dat"
#         )
        
        path_name = f"{OUTDIR}/path_neqs{NEQS}_lam6_{lam6:.10e}.dat"


        save_path(path, N, calc.npoints, path_name)

        print("\nDEBUG original calcpath end:")
        print("  original_end_index      =", getattr(calc, "original_end_index", None))
        print("  original_N_end          =", getattr(calc, "original_N_end", None))
        print("  original_N_before_end   =", getattr(calc, "original_N_before_end", None))
        print("  original_N_after_end    =", getattr(calc, "original_N_after_end", None))
        print("  original_eps_end        =", getattr(calc, "original_eps_end", None))
        print("  spectrum_N_start N[3]   =", N[3])
        print("  path_N_end N[-1]        =", N[calc.npoints - 1])

        summary_records.append({
            "accepted_index": accepted_count,
            "trial_index": trial_count,
            "lam5": lam5,
            "lam6": lam6,
            "r": r,
            "n_s": ns,
            "alpha_s": alpha_s,
            "Nefolds": calc.Nefolds,

            "original_end_index": getattr(calc, "original_end_index", np.nan),
            "original_N_end": getattr(calc, "original_N_end", np.nan),
            "original_N_before_end": getattr(calc, "original_N_before_end", np.nan),
            "original_N_after_end": getattr(calc, "original_N_after_end", np.nan),
            "original_eps_end": getattr(calc, "original_eps_end", np.nan),

            "spectrum_N_start": N[3],
            "path_N_end": N[calc.npoints - 1],

            "calc_ret": calc.ret,
            "outdir": OUTDIR,
        })

    summary_df = pd.DataFrame(summary_records)
    summary_file = f"{BASE_OUTDIR}/neqs{NEQS}_summary.csv"
    summary_df.to_csv(summary_file, index=False)

    print("\n" + "=" * 70)
    print("DONE")
    print(f"Accepted viable nontrivial models: {accepted_count}")
    print(f"Total trials: {trial_count}")
    print(f"Rejected asymptotes: {rejected_asymptote}")
    print(f"Rejected bad ns: {rejected_bad_ns}")
    print(f"Rejected other: {rejected_other}")
    print(f"Spectrum error count: {spectrum_error_count}")
    print(f"Duplicate directory count: {duplicate_dir_count}")
    print(f"Summary written to:\n{summary_file}")

    if accepted_count < TARGET_ACCEPTED:
        print(
            f"WARNING: only found {accepted_count}/{TARGET_ACCEPTED} accepted models "
            f"before hitting MAX_TRIALS={MAX_TRIALS}."
        )


%time run_neqs9_lam5_lam6_models()

Total λ5 trial pool: 101
Total λ6 trial pool: 101
Base λ5=-8.924610e-09 included: True
Base λ6=6.100000e-10 included: True
Removing old output directory:
/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs9

Trial 1 | accepted 0/5
Trying λ5 = -8.9246100000e-09
Trying λ6 = 6.1000000000e-10

DEBUG N values:
  N[0]   = 1.146444335769047e-06
  N[3]   = 0.00015614644433576904
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0305 s
calc.ret = nontrivial
Candidate observables:
  r       = 2.5291091526e-03
  ns      = 0.9706773294
  alpha_s = -4.3334678021e-04

*** ACCEPTED MODEL ***
accepted #1
λ5 = -8.9246100000e-09
λ6 = 6.1000000000e-10
ns = 0.9706773294
Evaluating spectrum for accepted model...
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72



DEBUG N values:
  N[0]   = 1.0146545744428294e-06
  N[3]   = 0.00015601465457444283
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0455 s
calc.ret = nontrivial
Candidate observables:
  r       = 8.8938315450e-08
  ns      = 0.5645986854
  alpha_s = 3.8928547013e-04
REJECTED: ns=0.5645986854 outside (0.96, 0.971)

Trial 20 | accepted 1/5
Trying λ5 = 1.1209572272e-05
Trying λ6 = 1.1693399687e-09

DEBUG N values:
  N[0]   = 999.999999
  N[3]   = 999.9998439999999
  N[end] = 0.0
  Nefolds target = 60
calcpath runtime: 0.0309 s
calc.ret = asymptote
REJECTED: asymptote

Trial 21 | accepted 1/5
Trying λ5 = 4.4374807851e-05
Trying λ6 = 1.8182029910e-09

DEBUG N values:
  N[0]   = 999.999999
  N[3]   = 999.9998439999999
  N[end] = 0.0
  Nefolds target = 60
calcpath runtime: 0.0355 s
calc.ret = asymptote
REJECTED: asymptote

Trial 22 | accepted 1/5
Trying λ5 = -1.4049209943e-05
Trying λ6 = -6.2968046201e-10

DEBUG N values:
  N[0]   = 1.0843559746499522e-06
  N[3]   = 0.0001560843559

DEBUG N values:
  N[0]   = 1.0207818402486736e-06
  N[3]   = 0.00015602078184024867
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0454 s
calc.ret = nontrivial
Candidate observables:
  r       = 1.5045251860e-07
  ns      = 0.5751796979
  alpha_s = 4.8683987997e-04
REJECTED: ns=0.5751796979 outside (0.96, 0.971)

Trial 44 | accepted 1/5
Trying λ5 = 6.6601454207e-06
Trying λ6 = -2.3461050906e-09

DEBUG N values:
  N[0]   = 999.999999
  N[3]   = 999.9998439999999
  N[end] = 0.0
  Nefolds target = 60
calcpath runtime: 0.0280 s
calc.ret = asymptote
REJECTED: asymptote

Trial 45 | accepted 1/5
Trying λ5 = 2.3248053467e-06
Trying λ6 = -4.0605948924e-09

DEBUG N values:
  N[0]   = 999.999999
  N[3]   = 999.9998439999999
  N[end] = 0.0
  Nefolds target = 60
calcpath runtime: 0.0247 s
calc.ret = asymptote
REJECTED: asymptote

Trial 46 | accepted 1/5
Trying λ5 = 7.5946495556e-06
Trying λ6 = 4.2929619758e-09

DEBUG N values:
  N[0]   = 999.999999
  N[3]   = 999.9998439999999
  N[end] =

DEBUG N values:
  N[0]   = 999.999999
  N[3]   = 999.9998439999999
  N[end] = 0.0
  Nefolds target = 60
calcpath runtime: 0.0286 s
calc.ret = asymptote
REJECTED: asymptote

Trial 71 | accepted 1/5
Trying λ5 = 1.5210327000e-05
Trying λ6 = -6.8581564566e-10

DEBUG N values:
  N[0]   = 999.999999
  N[3]   = 999.9998439999999
  N[end] = 0.0
  Nefolds target = 60
calcpath runtime: 0.0309 s
calc.ret = asymptote
REJECTED: asymptote

Trial 72 | accepted 1/5
Trying λ5 = 3.9654659585e-05
Trying λ6 = -1.3243812995e-09

DEBUG N values:
  N[0]   = 999.999999
  N[3]   = 999.9998439999999
  N[end] = 0.0
  Nefolds target = 60
calcpath runtime: 0.0357 s
calc.ret = asymptote
REJECTED: asymptote

Trial 73 | accepted 1/5
Trying λ5 = -6.4135074734e-06
Trying λ6 = 3.9192335502e-09

DEBUG N values:
  N[0]   = 1.0595744040765566e-06
  N[3]   = 0.00015605957440407655
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0420 s
calc.ret = nontrivial
Candidate observables:
  r       = 2.1163865136e-04
  ns  


DEBUG N values:
  N[0]   = 1.031462946026295e-06
  N[3]   = 0.0001560314629460263
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0418 s
calc.ret = nontrivial
Candidate observables:
  r       = 6.3434330104e-05
  ns      = 0.7371325306
  alpha_s = 5.0924927813e-03
REJECTED: ns=0.7371325306 outside (0.96, 0.971)

Trial 98 | accepted 1/5
Trying λ5 = -3.1380699412e-05
Trying λ6 = 4.4437238998e-09

DEBUG N values:
  N[0]   = 1.0127470229926984e-06
  N[3]   = 0.0001560127470229927
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0472 s
calc.ret = nontrivial
Candidate observables:
  r       = 7.7813725626e-07
  ns      = 0.6098565818
  alpha_s = 9.6878727617e-04
REJECTED: ns=0.6098565818 outside (0.96, 0.971)

Trial 99 | accepted 1/5
Trying λ5 = 2.3955079505e-05
Trying λ6 = -9.5411913824e-11

DEBUG N values:
  N[0]   = 999.999999
  N[3]   = 999.9998439999999
  N[end] = 0.0
  Nefolds target = 60
calcpath runtime: 0.0335 s
calc.ret = asymptote
REJECTED: asymptote

Trial 100

...


DEBUG N values:
  N[0]   = 1.0396956920667435e-06
  N[3]   = 0.00015603969569206674
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0450 s
calc.ret = nontrivial
Candidate observables:
  r       = 6.3504256567e-07
  ns      = 0.6053519397
  alpha_s = 8.9092442240e-04
REJECTED: ns=0.6053519397 outside (0.96, 0.971)

Trial 867 | accepted 3/5
Trying λ5 = -3.9031693789e-05
Trying λ6 = -1.7830238154e-09

DEBUG N values:
  N[0]   = 1.0695936250849627e-06
  N[3]   = 0.00015606959362508496
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0481 s
calc.ret = nontrivial
Candidate observables:
  r       = 2.6552098925e-07
  ns      = 0.5868117686
  alpha_s = 6.1896416590e-04
REJECTED: ns=0.5868117686 outside (0.96, 0.971)

Trial 868 | accepted 3/5
Trying λ5 = -7.3406090401e-06
Trying λ6 = -4.7545188348e-09

DEBUG N values:
  N[0]   = 1.0287329839920857e-06
  N[3]   = 0.00015602873298399208
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0433 s
calc.ret = nontrivial
Cand

DEBUG N values:
  N[0]   = 999.999999
  N[3]   = 999.9998439999999
  N[end] = 0.0
  Nefolds target = 60
calcpath runtime: 0.0303 s
calc.ret = asymptote
REJECTED: asymptote

Trial 888 | accepted 4/5
Trying λ5 = -3.0615644855e-06
Trying λ6 = 2.5945025146e-09

DEBUG N values:
  N[0]   = 1.0203253875952214e-06
  N[3]   = 0.00015602032538759522
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0380 s
calc.ret = nontrivial
Candidate observables:
  r       = 7.0086067816e-04
  ns      = 0.8864149364
  alpha_s = 6.3403348738e-03
REJECTED: ns=0.8864149364 outside (0.96, 0.971)

Trial 889 | accepted 4/5
Trying λ5 = -3.2179904506e-05
Trying λ6 = -3.2882795184e-09

DEBUG N values:
  N[0]   = 1.0222572680286248e-06
  N[3]   = 0.00015602225726802862
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0445 s
calc.ret = nontrivial
Candidate observables:
  r       = 6.8839329562e-07
  ns      = 0.6070786955
  alpha_s = 9.2137653979e-04
REJECTED: ns=0.6070786955 outside (0.96, 0.971)

Tria


DEBUG N values:
  N[0]   = 1.0234877006732858e-06
  N[3]   = 0.00015602348770067328
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0454 s
calc.ret = nontrivial
Candidate observables:
  r       = 2.3834314734e-07
  ns      = 0.5845875458
  alpha_s = 5.9140807933e-04
REJECTED: ns=0.5845875458 outside (0.96, 0.971)

Trial 917 | accepted 4/5
Trying λ5 = -1.8630494598e-05
Trying λ6 = 1.2408482025e-10

DEBUG N values:
  N[0]   = 1.0590379158893483e-06
  N[3]   = 0.00015605903791588935
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0423 s
calc.ret = nontrivial
Candidate observables:
  r       = 7.6252729221e-06
  ns      = 0.6656197962
  alpha_s = 2.4160422839e-03
REJECTED: ns=0.6656197962 outside (0.96, 0.971)

Trial 918 | accepted 4/5
Trying λ5 = -1.9829842536e-05
Trying λ6 = 3.6182299198e-09

DEBUG N values:
  N[0]   = 1.0248396645474713e-06
  N[3]   = 0.00015602483966454747
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0440 s
calc.ret = nontrivial
Candid


DEBUG N values:
  N[0]   = 1.0707588014847715e-06
  N[3]   = 0.00015607075880148477
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0488 s
calc.ret = nontrivial
Candidate observables:
  r       = 1.7050108389e-05
  ns      = 0.6893710847
  alpha_s = 3.2667313000e-03
REJECTED: ns=0.6893710847 outside (0.96, 0.971)

Trial 939 | accepted 4/5
Trying λ5 = -8.6865228217e-06
Trying λ6 = 2.2824282924e-10

DEBUG N values:
  N[0]   = 1.0185375483852112e-06
  N[3]   = 0.0001560185375483852
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0479 s
calc.ret = nontrivial
Candidate observables:
  r       = 1.0003380368e-04
  ns      = 0.7576455523
  alpha_s = 5.7947804262e-03
REJECTED: ns=0.7576455523 outside (0.96, 0.971)

Trial 940 | accepted 4/5
Trying λ5 = -4.5555661174e-05
Trying λ6 = -3.5415883413e-09

DEBUG N values:
  N[0]   = 1.0199285068447352e-06
  N[3]   = 0.00015601992850684473
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0496 s
calc.ret = nontrivial
Candid


DEBUG N values:
  N[0]   = 1.0122577148431446e-06
  N[3]   = 0.00015601225771484314
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0481 s
calc.ret = nontrivial
Candidate observables:
  r       = 7.7520822307e-07
  ns      = 0.6096684271
  alpha_s = 9.6785730270e-04
REJECTED: ns=0.6096684271 outside (0.96, 0.971)

Trial 964 | accepted 4/5
Trying λ5 = 4.2775307932e-05
Trying λ6 = 3.7140419083e-10

DEBUG N values:
  N[0]   = 999.999999
  N[3]   = 999.9998439999999
  N[end] = 0.0
  Nefolds target = 60
calcpath runtime: 0.0380 s
calc.ret = asymptote
REJECTED: asymptote

Trial 965 | accepted 4/5
Trying λ5 = -4.0755181975e-05
Trying λ6 = 3.4292111213e-09

DEBUG N values:
  N[0]   = 1.0188942976819817e-06
  N[3]   = 0.00015601889429768198
  N[end] = 60.0
  Nefolds target = 60
calcpath runtime: 0.0482 s
calc.ret = nontrivial
Candidate observables:
  r       = 2.1322602367e-07
  ns      = 0.5823201870
  alpha_s = 5.6418556669e-04
REJECTED: ns=0.5823201870 outside (0.96, 0.971)

Trial